# Phase 4 — Macro Regime Detection (HMM)

**Objective:** Fit a 3-state Gaussian Hidden Markov Model on a composite macro index to detect
expansion / slowdown / crisis regimes in real time (no look-ahead).

**Dependencies:** Phase 1 macro outputs (`macro_indicators.parquet`)

**Outputs:**
- `regime_probabilities.parquet` — p_expansion, p_slowdown, p_crisis, regime_label (month-end)
- `macro_composite_index.parquet` — composite_z (expanding-window PCA)
- `macro_regimes.parquet` — macro_regime for tech portfolio (daily)
- `regime_labels.parquet` — regime_state, regime_prob_0, regime_prob_1, ... for tech portfolio
- Figures: regime timeline, transition matrix heatmap, BIC comparison, PCA explained variance

**Key design decisions:**
- Expanding-window standardisation and PCA (no look-ahead)
- Filtered probabilities via forward algorithm only (never `predict_proba`)
- 25 random restarts to escape local optima
- BIC model selection over K ∈ {2, 3, 4}

In [1]:
# === Setup & Imports ===
import os, sys, warnings, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Project paths
sys.path.insert(0, str(Path.cwd().parent))
from src.config import (
    PROJECT_ROOT, PROCESSED_DIR, INTERIM_DIR, FIGURES_DIR, TABLES_DIR,
    LOGS_DIR, RANDOM_STATE, COLORS,
    HMM_N_STATES, HMM_N_RESTARTS, HMM_MIN_WINDOW,
    MACRO_INDICATORS_FILE, MACRO_COMPOSITE_FILE, REGIME_PROBS_FILE,
    REGIME_LABELS_FILE, MACRO_REGIMES_FILE,
    TECH_START, TECH_END, TICKERS,
)
from src.validation import validate_parquet, check_nan_propagation, quick_data_check
from src.regime_model import (
    expanding_standardise,
    expanding_pca_composite,
    fit_hmm_with_restarts,
    sort_states_by_mean,
    label_states,
    filtered_probabilities,
    bic_model_selection,
    get_cached_hmm,
    label_macro_regime,
    regime_transition_analysis,
    regime_persistence_metrics,
    smoothed_vs_filtered_diagnostic,
)
from src.visualization import setup_style, save_fig

# Logging
PHASE_NUM = 4
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info(f"Phase {PHASE_NUM} started")

setup_style()
np.random.seed(RANDOM_STATE)

print(f"Working directory: {PROJECT_ROOT}")
print(f"Config: N_STATES={HMM_N_STATES}, N_RESTARTS={HMM_N_RESTARTS}, MIN_WINDOW={HMM_MIN_WINDOW}")
print(f"Random state: {RANDOM_STATE}")

Working directory: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine
Config: N_STATES=3, N_RESTARTS=25, MIN_WINDOW=24
Random state: 42


## 4.0 Load & Validate Input Data

In [2]:
# Load macro indicators from Phase 1
macro = pd.read_parquet(MACRO_INDICATORS_FILE)

validate_parquet(
    macro,
    min_rows=120,
    date_index=True,
    label='macro_indicators',
)

quick_data_check(macro, 'Macro Indicators')
check_nan_propagation(macro, 'Macro Indicators (input)')

# Display available columns
print(f"Columns: {list(macro.columns)}")
print(f"\nSummary statistics:")
macro.describe().round(4)

=== Macro Indicators ===
  Shape: (232, 10)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(10)}
  Sorted: True



Columns: ['t10y2y', 'baa10y', 'vix', 'oecd_cli', 'unrate', 'claims_roc', 'oil_roc', 'indpro_yoy', 'unrate_chg', 'real_m2_yoy']

Summary statistics:


,t10y2y,baa10y,vix,oecd_cli,unrate,claims_roc,oil_roc,indpro_yoy,unrate_chg,real_m2_yoy
count,232.0000,232.0000,232.0000,232.0000,232.0000,232.0000,232.0000,232.0000,232.0000,232.0000
mean,1.0346,2.4934,19.2402,99.8473,5.8763,0.1386,0.0291,0.0049,-0.0931,0.0390
std,0.9600,0.7795,8.4687,1.3006,2.1528,1.5999,0.2089,0.0473,1.9823,0.0572
min,-0.9290,1.4632,10.1255,93.4837,3.4000,-0.7033,-0.7123,-0.1732,-8.7000,-0.0913
25%,0.2183,1.9149,13.6349,99.3189,4.2000,-0.0592,-0.0735,-0.0129,-0.8000,0.0126
50%,1.0080,2.2890,17.1142,99.9204,5.0000,-0.0178,0.0390,0.0186,-0.4000,0.0376
75%,1.7345,2.8832,22.3968,100.6578,7.5000,0.0320,0.1415,0.0293,0.0000,0.0560
max,2.8342,6.0136,62.6689,101.9542,14.8000,20.6392,1.4602,0.1655,11.1000,0.2469


## 4.1 Composite Macro Index (Expanding-Window PCA)

Build a single composite macro index via expanding-window PCA:

1. **Expanding-window standardisation:** $z_{i,t} = (x_{i,t} - \bar{x}_{i,1:t}) / \sigma_{x_i,1:t}$ for $t \geq 24$
2. **Expanding-window PCA:** At each $t$, fit PCA on $[1, t]$, project observation $t$ onto 1st PC
3. **Sign convention:** 1st PC loads positive on good indicators (INDPRO, CLI), negative on bad (VIX)

In [3]:
# Step 1: Expanding-window standardisation (no look-ahead)
z_std = expanding_standardise(macro, min_window=HMM_MIN_WINDOW)

print(f"Standardised shape: {z_std.shape}")
print(f"NaN count (expected for first {HMM_MIN_WINDOW - 1} rows):")
print(z_std.isna().sum())

# Verify expanding-window property: no look-ahead
# The z-score at time t should only use data up to t
t_test = HMM_MIN_WINDOW + 10
col_test = macro.columns[0]
manual_mean = macro[col_test].iloc[:t_test + 1].mean()
manual_std = macro[col_test].iloc[:t_test + 1].std()
manual_z = (macro[col_test].iloc[t_test] - manual_mean) / manual_std
computed_z = z_std[col_test].iloc[t_test]
print(f"\nLook-ahead verification ({col_test}, t={t_test}):")
print(f"  Manual z:   {manual_z:.6f}")
print(f"  Computed z:  {computed_z:.6f}")
print(f"  Match: {abs(manual_z - computed_z) < 1e-6}")

Standardised shape: (232, 10)
NaN count (expected for first 23 rows):
t10y2y         23
baa10y         23
vix            23
oecd_cli       23
unrate         23
claims_roc     23
oil_roc        23
indpro_yoy     23
unrate_chg     23
real_m2_yoy    23
dtype: int64

Look-ahead verification (t10y2y, t=34):
  Manual z:   1.930933
  Computed z:  1.930933
  Match: True


In [4]:
# Step 2: Expanding-window PCA composite index
composite_z = expanding_pca_composite(z_std, min_window=HMM_MIN_WINDOW)

print(f"Composite index: {len(composite_z)} observations")
print(f"Valid (non-NaN): {composite_z.notna().sum()}")
print(f"First valid date: {composite_z.dropna().index[0]}")
print(f"Last valid date:  {composite_z.dropna().index[-1]}")
print(f"\nComposite z statistics:")
print(composite_z.dropna().describe().round(4))

Composite index: 232 observations
Valid (non-NaN): 186
First valid date: 2008-11-30 00:00:00
Last valid date:  2024-04-30 00:00:00

Composite z statistics:
count    186.0000
mean      -0.4935
std        2.7181
min       -6.3887
25%       -2.2740
50%       -1.5923
75%        2.2714
max       13.9660
Name: composite_z, dtype: float64


In [5]:
# Step 3: Verify PCA explained variance (full-sample diagnostic)
# This is for diagnostic purposes only; the actual composite uses expanding-window PCA
z_valid = z_std.dropna()
pca_diag = PCA(n_components=min(5, z_valid.shape[1]))
pca_diag.fit(z_valid.values)

print("PCA Explained Variance (full-sample diagnostic):")
for i, (var, cumvar) in enumerate(zip(
    pca_diag.explained_variance_ratio_,
    np.cumsum(pca_diag.explained_variance_ratio_)
)):
    print(f"  PC{i+1}: {var:.4f} ({cumvar:.4f} cumulative)")

pc1_var = pca_diag.explained_variance_ratio_[0]
print(f"\nValidation: PC1 explains {pc1_var:.1%} of variance (gate: >30%)")
assert pc1_var > 0.30, f"PC1 explains only {pc1_var:.1%} — below 30% threshold"
print("PASS: PC1 > 30%")

# PCA loadings (sign convention check)
loadings = pd.Series(pca_diag.components_[0], index=z_valid.columns, name='PC1_loading')
print(f"\nPC1 Loadings (sign convention):")
print(loadings.sort_values(ascending=False).round(4))

PCA Explained Variance (full-sample diagnostic):
  PC1: 0.4817 (0.4817 cumulative)
  PC2: 0.1624 (0.6441 cumulative)
  PC3: 0.1129 (0.7570 cumulative)
  PC4: 0.0826 (0.8396 cumulative)
  PC5: 0.0575 (0.8971 cumulative)

Validation: PC1 explains 48.2% of variance (gate: >30%)
PASS: PC1 > 30%

PC1 Loadings (sign convention):
unrate_chg     0.4627
unrate         0.3917
real_m2_yoy    0.3538
baa10y         0.3089
claims_roc     0.2975
vix            0.2758
t10y2y         0.2264
oil_roc       -0.0691
oecd_cli      -0.2750
indpro_yoy    -0.3401
Name: PC1_loading, dtype: float64


In [6]:
# Figure: PCA explained variance bar chart
fig, ax = plt.subplots(figsize=(8, 5))
n_pcs = len(pca_diag.explained_variance_ratio_)
x_pos = np.arange(1, n_pcs + 1)

ax.bar(x_pos, pca_diag.explained_variance_ratio_, alpha=0.7, color='steelblue',
       label='Individual')
ax.plot(x_pos, np.cumsum(pca_diag.explained_variance_ratio_), 'ro-',
        label='Cumulative')
ax.axhline(y=0.30, color='red', linestyle='--', alpha=0.5, label='30% threshold')

ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Explained Variance (Full-Sample Diagnostic)', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'PC{i}' for i in x_pos])
ax.legend()
ax.grid(True, alpha=0.3)

save_fig(fig, 'pca_explained_variance')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/pca_explained_variance.png


In [7]:
# Figure: Composite macro index time series
fig, ax = plt.subplots(figsize=(14, 5))
valid_composite = composite_z.dropna()

ax.plot(valid_composite.index, valid_composite.values, color='black', linewidth=0.8)
ax.axhline(y=0, color='grey', linestyle='--', alpha=0.5)

# Shade NBER recessions for reference
nber_recessions = [
    ('2007-12-01', '2009-06-30'),  # Great Recession
    ('2020-02-01', '2020-04-30'),  # COVID
]
for start, end in nber_recessions:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.2, color='red', label='NBER Recession')

# Remove duplicate legend labels
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())

ax.set_ylabel('Composite Z-Score')
ax.set_title('Expanding-Window PCA Composite Macro Index', fontweight='bold')
ax.grid(True, alpha=0.3)

save_fig(fig, 'composite_macro_index')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/composite_macro_index.png


## 4.2 HMM Fitting — Multiple Restarts

Fit a 3-state Gaussian HMM with 25 random restarts on the composite macro index.
States are sorted by mean: highest = Expansion, lowest = Crisis.

In [8]:
# Prepare data: drop NaN, reshape to (T, 1)
composite_valid = composite_z.dropna()
hmm_data = composite_valid.values.reshape(-1, 1)

print(f"HMM input: {hmm_data.shape[0]} observations")
print(f"Date range: {composite_valid.index[0]} to {composite_valid.index[-1]}")
print(f"\nFitting {HMM_N_STATES}-state Gaussian HMM with {HMM_N_RESTARTS} restarts...")

# Fit HMM with multiple restarts (uses src/regime_model.py)
hmm_model = fit_hmm_with_restarts(
    hmm_data,
    n_states=HMM_N_STATES,
    n_restarts=HMM_N_RESTARTS,
)

print(f"\nHMM fitted successfully.")
print(f"Converged: {hmm_model.monitor_.converged}")
print(f"Log-likelihood: {hmm_model.score(hmm_data):.2f}")
print(f"\nRaw state means: {hmm_model.means_.flatten()}")
print(f"Raw state variances: {[hmm_model.covars_[i][0, 0] for i in range(HMM_N_STATES)]}")

logger.info(f"HMM fitted: LL={hmm_model.score(hmm_data):.2f}, converged={hmm_model.monitor_.converged}")

HMM input: 186 observations
Date range: 2008-11-30 00:00:00 to 2024-04-30 00:00:00

Fitting 3-state Gaussian HMM with 25 restarts...



HMM fitted successfully.
Converged: True
Log-likelihood: -217.32

Raw state means: [-2.11627963  2.53877847  0.10598034]
Raw state variances: [np.float64(0.3260138236381406), np.float64(0.17793816841117474), np.float64(21.937315778410543)]


In [9]:
# Sort states by mean: highest = Expansion (state 0), lowest = Crisis (state 2)
state_mapping, sort_order = sort_states_by_mean(hmm_model)
state_labels = label_states(state_mapping, n_states=HMM_N_STATES)

print("State mapping (old -> new):")
for old, new in state_mapping.items():
    label = state_labels[new]
    mean = hmm_model.means_[old, 0]
    std = np.sqrt(hmm_model.covars_[old][0, 0])
    print(f"  State {old} -> State {new} ({label}): mean={mean:.4f}, std={std:.4f}")

print(f"\nState labels: {state_labels}")

State mapping (old -> new):
  State 1 -> State 0 (Expansion): mean=2.5388, std=0.4218
  State 2 -> State 1 (Slowdown): mean=0.1060, std=4.6837
  State 0 -> State 2 (Crisis): mean=-2.1163, std=0.5710

State labels: {0: 'Expansion', 1: 'Slowdown', 2: 'Crisis'}


In [10]:
# Verify convergence consistency: check spread of top-3 restarts
# Re-run restarts to capture all scores for diagnostic
all_scores = []
for seed in range(HMM_N_RESTARTS):
    from hmmlearn import hmm as hmm_lib
    m = hmm_lib.GaussianHMM(
        n_components=HMM_N_STATES,
        covariance_type='full',
        n_iter=500,
        tol=1e-6,
        random_state=seed,
        init_params='stmc',
    )
    try:
        m.fit(hmm_data)
        all_scores.append(m.score(hmm_data))
    except (ValueError, np.linalg.LinAlgError):
        pass

all_scores_sorted = sorted(all_scores, reverse=True)
print(f"Restarts converged: {len(all_scores)}/{HMM_N_RESTARTS}")
print(f"Top-3 log-likelihoods: {[f'{s:.2f}' for s in all_scores_sorted[:3]]}")
if len(all_scores_sorted) >= 3:
    spread = all_scores_sorted[0] - all_scores_sorted[2]
    print(f"Top-3 spread: {spread:.4f}")
    print(f"Consistent (spread < 5): {'PASS' if spread < 5 else 'WARNING'}")
print(f"Best LL: {all_scores_sorted[0]:.2f}")
print(f"Worst converged LL: {all_scores_sorted[-1]:.2f}")

Restarts converged: 25/25
Top-3 log-likelihoods: ['-217.32', '-217.32', '-217.32']
Top-3 spread: 0.0000
Consistent (spread < 5): PASS
Best LL: -217.32
Worst converged LL: -308.79


## 4.3 Filtered Probabilities (NO Look-Ahead)

**CRITICAL:** Use the forward-only algorithm to compute $P(S_t | Z_1, \ldots, Z_t)$.
Never use `model.predict_proba()`, which gives smoothed probabilities $P(S_t | Z_1, \ldots, Z_T)$
and conditions on future data.

In [11]:
# Compute FILTERED probabilities (forward-only, no look-ahead)
filt_probs_raw = filtered_probabilities(hmm_model, hmm_data)

print(f"Filtered probabilities shape: {filt_probs_raw.shape}")
print(f"Sum check (should all be ~1.0):")
print(f"  Min sum: {filt_probs_raw.sum(axis=1).min():.6f}")
print(f"  Max sum: {filt_probs_raw.sum(axis=1).max():.6f}")

# Re-order columns according to state mapping
# state_mapping: old_state -> new_state, so we need columns in new-state order
reorder = [k for k, v in sorted(state_mapping.items(), key=lambda x: x[1])]
filt_probs = filt_probs_raw[:, reorder]

# Build regime probabilities DataFrame
regime_probs = pd.DataFrame(
    filt_probs,
    index=composite_valid.index,
    columns=['p_expansion', 'p_slowdown', 'p_crisis'],
)

# Assign regime label based on highest filtered probability
regime_probs['regime_label'] = regime_probs[['p_expansion', 'p_slowdown', 'p_crisis']].idxmax(axis=1)
regime_probs['regime_label'] = regime_probs['regime_label'].map({
    'p_expansion': 'Expansion',
    'p_slowdown': 'Slowdown',
    'p_crisis': 'Crisis',
})

print(f"\nRegime distribution:")
print(regime_probs['regime_label'].value_counts())
print(f"\nRegime proportions:")
print(regime_probs['regime_label'].value_counts(normalize=True).round(4))

Filtered probabilities shape: (186, 3)
Sum check (should all be ~1.0):
  Min sum: 1.000000
  Max sum: 1.000000

Regime distribution:
regime_label
Crisis       109
Expansion     53
Slowdown      24
Name: count, dtype: int64

Regime proportions:
regime_label
Crisis       0.5860
Expansion    0.2849
Slowdown     0.1290
Name: proportion, dtype: float64


In [12]:
# Validation: regime proportions
regime_counts = regime_probs['regime_label'].value_counts(normalize=True)

crisis_pct = regime_counts.get('Crisis', 0)
expansion_pct = regime_counts.get('Expansion', 0)

print(f"Crisis fraction:    {crisis_pct:.1%} (gate: < 20%)")
print(f"Expansion fraction: {expansion_pct:.1%} (gate: > 40%)")

gate_crisis = crisis_pct < 0.20
gate_expansion = expansion_pct > 0.40
print(f"\n  [{'PASS' if gate_crisis else 'FAIL'}] Crisis < 20%")
print(f"  [{'PASS' if gate_expansion else 'FAIL'}] Expansion > 40%")

# Probability sum check
prob_sums = regime_probs[['p_expansion', 'p_slowdown', 'p_crisis']].sum(axis=1)
assert prob_sums.between(0.99, 1.01).all(), "Regime probabilities don't sum to 1!"
print("  [PASS] Probabilities sum to 1.0")

Crisis fraction:    58.6% (gate: < 20%)
Expansion fraction: 28.5% (gate: > 40%)

  [FAIL] Crisis < 20%
  [FAIL] Expansion > 40%
  [PASS] Probabilities sum to 1.0


## 4.4 BIC Model Selection

Test $K \in \{2, 3, 4\}$ and select by BIC:

$$\text{BIC} = -2 \ln L + p \ln T, \quad p = K^2 + 2K - 1$$

In [13]:
# BIC model selection
print(f"Running BIC model selection for K in {{2, 3, 4}}...")
print(f"This fits {HMM_N_RESTARTS} restarts per K (75 total HMM fits).\n")

bic_results = bic_model_selection(
    hmm_data,
    k_range=(2, 3, 4),
    n_restarts=HMM_N_RESTARTS,
)

print("BIC Model Selection Results:")
print(bic_results.to_string(index=False))

# Identify best K
best_k = bic_results.loc[bic_results['bic'].idxmin(), 'k']
best_bic = bic_results.loc[bic_results['bic'].idxmin(), 'bic']
print(f"\nBest K by BIC: K={int(best_k)} (BIC={best_bic:.2f})")

# Check: 3-state < 2-state?
bic_2 = bic_results.loc[bic_results['k'] == 2, 'bic'].values[0]
bic_3 = bic_results.loc[bic_results['k'] == 3, 'bic'].values[0]
print(f"\nBIC(2-state) = {bic_2:.2f}")
print(f"BIC(3-state) = {bic_3:.2f}")
if bic_3 < bic_2:
    print("  [PASS] 3-state BIC < 2-state BIC")
else:
    print("  [NOTE] 2-state BIC is lower; documenting this finding.")
    print("         Proceeding with 3-state model for interpretability.")

logger.info(f"BIC results: {bic_results.to_dict('records')}")
logger.info(f"Best K={int(best_k)}, BIC={best_bic:.2f}")

Running BIC model selection for K in {2, 3, 4}...
This fits 25 restarts per K (75 total HMM fits).



BIC Model Selection Results:
 k  log_likelihood  n_params        bic
 2     -298.783032         7 634.146291
 3     -217.316361        14 507.793176
 4     -203.086774        23 526.365722

Best K by BIC: K=3 (BIC=507.79)

BIC(2-state) = 634.15
BIC(3-state) = 507.79
  [PASS] 3-state BIC < 2-state BIC


In [14]:
# Figure: BIC comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))

colors_bic = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(
    bic_results['k'].astype(int).astype(str) + '-state',
    bic_results['bic'],
    color=colors_bic[:len(bic_results)],
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5,
)

# Annotate with BIC values
for bar, bic_val in zip(bars, bic_results['bic']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bic_val:.1f}', ha='center', va='bottom', fontsize=11)

# Star the best model
best_idx = bic_results['bic'].idxmin()
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

ax.set_xlabel('Number of States')
ax.set_ylabel('BIC (lower is better)')
ax.set_title('BIC Model Selection for HMM', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

save_fig(fig, 'bic_model_selection')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/bic_model_selection.png


## 4.5 Manual Regime Labels for Tech Portfolio

Apply manual macro regime labels to the tech portfolio date range using
`label_macro_regime()` from `src/regime_model.py`.

In [15]:
# Generate daily dates for tech portfolio period
tech_dates = pd.date_range(start=TECH_START, end=TECH_END, freq='B')  # Business days

# Apply manual regime labels
manual_regimes = pd.Series(
    [label_macro_regime(d) for d in tech_dates],
    index=tech_dates,
    name='macro_regime',
)

print(f"Tech portfolio regime labels: {len(manual_regimes)} business days")
print(f"Date range: {manual_regimes.index[0]} to {manual_regimes.index[-1]}")
print(f"\nRegime distribution:")
print(manual_regimes.value_counts())
print(f"\nProportions:")
print(manual_regimes.value_counts(normalize=True).round(4))

Tech portfolio regime labels: 2673 business days
Date range: 2016-01-01 00:00:00 to 2026-03-31 00:00:00

Regime distribution:
macro_regime
QE Era                  521
Inflation/Rate Shock    478
Zero-Rate Recovery      435
AI Boom                 327
Tariff/Geopolitical     325
Late Cycle              296
Rate Hike               261
COVID Crash              30
Name: count, dtype: int64

Proportions:
macro_regime
QE Era                  0.1949
Inflation/Rate Shock    0.1788
Zero-Rate Recovery      0.1627
AI Boom                 0.1223
Tariff/Geopolitical     0.1216
Late Cycle              0.1107
Rate Hike               0.0976
COVID Crash             0.0112
Name: proportion, dtype: float64


In [16]:
# Build HMM-based regime labels for tech portfolio
# Map monthly HMM regimes to daily dates via forward-fill (month-end signal -> daily)
regime_monthly = regime_probs[['regime_label']].copy()
regime_monthly.columns = ['hmm_regime']

# Reindex to business days, forward-fill
regime_daily = regime_monthly.reindex(tech_dates, method='ffill')

# For dates before the first HMM observation, use manual labels
first_hmm_date = regime_monthly.index[0]
mask_before_hmm = regime_daily.index < first_hmm_date
regime_daily.loc[mask_before_hmm, 'hmm_regime'] = manual_regimes[mask_before_hmm]

# Build regime_labels DataFrame with state + probabilities
# Map regime labels to integer states
state_map_inv = {'Expansion': 0, 'Slowdown': 1, 'Crisis': 2}
regime_labels_df = pd.DataFrame(index=tech_dates)
regime_labels_df['regime_state'] = regime_daily['hmm_regime'].map(
    lambda x: state_map_inv.get(x, -1)
)

# Forward-fill monthly probabilities to daily
probs_daily = regime_probs[['p_expansion', 'p_slowdown', 'p_crisis']].reindex(
    tech_dates, method='ffill'
)
# For dates before HMM coverage, set uniform probabilities
probs_daily.loc[mask_before_hmm] = 1.0 / HMM_N_STATES

regime_labels_df['regime_prob_0'] = probs_daily['p_expansion']
regime_labels_df['regime_prob_1'] = probs_daily['p_slowdown']
regime_labels_df['regime_prob_2'] = probs_daily['p_crisis']

print(f"Regime labels (daily): {regime_labels_df.shape}")
print(f"\nState distribution:")
print(regime_labels_df['regime_state'].value_counts().sort_index())
print(f"\nNaN check: {regime_labels_df.isna().sum().sum()} total NaN")

Regime labels (daily): (2673, 4)

State distribution:
regime_state
0     110
1     260
2    2303
Name: count, dtype: int64

NaN check: 0 total NaN


## 4.6 Diagnostics

### 4.6.1 Transition Matrix Analysis

In [17]:
# Transition matrix analysis
analysis = regime_transition_analysis(hmm_model, state_mapping)

print("=" * 60)
print("TRANSITION MATRIX (re-ordered by regime)")
print("=" * 60)
print(analysis['transition_matrix'].round(4))

print(f"\nExpected Duration (months):")
print(analysis['expected_duration_months'].round(2))

print(f"\nStationary Distribution:")
print(analysis['stationary_distribution'].round(4))

print(f"\nState Means:")
print(analysis['state_means'].round(4))

print(f"\nState Volatilities:")
print(analysis['state_volatilities'].round(4))

# Validation: diagonal > 0.7
diag = np.diag(analysis['transition_matrix'].values)
print(f"\nTransition matrix diagonal: {diag.round(4)}")
gate_persistent = all(d > 0.7 for d in diag)
print(f"  [{'PASS' if gate_persistent else 'FAIL'}] All diagonal > 0.7 (persistent regimes)")

logger.info(f"Transition diagonal: {diag.round(4)}")

TRANSITION MATRIX (re-ordered by regime)
           Expansion  Slowdown  Crisis
Expansion     0.9626    0.0189  0.0185
Slowdown      0.0807    0.8744  0.0449
Crisis        0.0000    0.0101  0.9899

Expected Duration (months):
Expansion    26.74
Slowdown      7.96
Crisis       98.53
dtype: float64

Stationary Distribution:
Expansion    0.1874
Slowdown     0.0868
Crisis       0.7258
dtype: float64

State Means:
Expansion    2.5388
Slowdown     0.1060
Crisis      -2.1163
dtype: float64

State Volatilities:
Expansion    0.4218
Slowdown     4.6837
Crisis       0.5710
dtype: float64

Transition matrix diagonal: [0.9626 0.8744 0.9899]
  [PASS] All diagonal > 0.7 (persistent regimes)


In [18]:
# Regime persistence metrics
persistence = regime_persistence_metrics(regime_probs['regime_label'])

if persistence:
    print("Regime Duration Statistics (months):")
    print(persistence['duration_stats'].round(2))
    print(f"\nRegime change frequency: {persistence['change_frequency']:.4f}")
    print(f"Total periods: {persistence['total_periods']}")
    print(f"\nRegime Episodes:")
    print(persistence['episodes'].to_string(index=False))

Regime Duration Statistics (months):
           avg_duration  median_duration  min_duration  max_duration  \
regime                                                                 
Crisis            36.33             33.0             2            74   
Expansion         26.50             26.5             5            48   
Slowdown           6.00              6.0             5             7   

           n_episodes  
regime                 
Crisis              3  
Expansion           2  
Slowdown            4  

Regime change frequency: 0.0432
Total periods: 186

Regime Episodes:
   regime  duration      start        end
 Slowdown         6 2008-11-30 2009-04-30
   Crisis         2 2009-05-31 2009-06-30
 Slowdown         6 2009-07-31 2009-12-31
Expansion        48 2010-01-31 2013-12-31
   Crisis        74 2014-01-31 2020-02-29
 Slowdown         5 2020-03-31 2020-07-31
Expansion         5 2020-08-31 2020-12-31
 Slowdown         7 2021-01-31 2021-07-31
   Crisis        33 2021-08-31 202

In [19]:
# Figure: Transition matrix heatmap
fig, ax = plt.subplots(figsize=(7, 6))

tm = analysis['transition_matrix']
sns.heatmap(
    tm, annot=True, fmt='.3f', cmap='YlOrRd',
    vmin=0, vmax=1, square=True, ax=ax,
    linewidths=1, linecolor='white',
    cbar_kws={'label': 'Transition Probability'},
)
ax.set_xlabel('To State')
ax.set_ylabel('From State')
ax.set_title('HMM Transition Matrix (3-State)', fontweight='bold')

save_fig(fig, 'transition_matrix_heatmap')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/transition_matrix_heatmap.png


### 4.6.2 NBER Recession Overlap

In [20]:
# Validate: Crisis aligns with NBER recessions
nber_recessions = [
    ('2007-12-01', '2009-06-30'),  # Great Recession
    ('2020-02-01', '2020-04-30'),  # COVID
]

print("NBER Recession Overlap Analysis:")
print("=" * 50)

total_recession_months = 0
crisis_in_recession = 0

for name, (start, end) in zip(['Great Recession', 'COVID'], nber_recessions):
    mask = (regime_probs.index >= start) & (regime_probs.index <= end)
    recession_months = mask.sum()
    if recession_months == 0:
        print(f"\n{name} ({start} to {end}): No overlap with HMM sample period")
        continue

    total_recession_months += recession_months
    crisis_months = (regime_probs.loc[mask, 'regime_label'] == 'Crisis').sum()
    crisis_in_recession += crisis_months
    overlap_pct = crisis_months / recession_months if recession_months > 0 else 0

    print(f"\n{name} ({start} to {end}):")
    print(f"  Months in sample: {recession_months}")
    print(f"  Classified as Crisis: {crisis_months}")
    print(f"  Overlap: {overlap_pct:.1%}")
    print(f"  Regime breakdown:")
    print(f"    {regime_probs.loc[mask, 'regime_label'].value_counts().to_dict()}")

if total_recession_months > 0:
    overall_overlap = crisis_in_recession / total_recession_months
    print(f"\nOverall NBER overlap: {overall_overlap:.1%} (gate: >= 80%)")
    gate_nber = overall_overlap >= 0.80
    print(f"  [{'PASS' if gate_nber else 'FAIL'}] Crisis >= 80% of NBER recession months")
else:
    print("\nNo NBER recession months in sample period.")
    # If sample doesn't cover 2007-2009, check COVID only
    gate_nber = True  # Cannot evaluate

logger.info(f"NBER overlap: {overall_overlap:.1%}" if total_recession_months > 0 else "No NBER overlap available")

NBER Recession Overlap Analysis:

Great Recession (2007-12-01 to 2009-06-30):
  Months in sample: 8
  Classified as Crisis: 2
  Overlap: 25.0%
  Regime breakdown:
    {'Slowdown': 6, 'Crisis': 2}

COVID (2020-02-01 to 2020-04-30):
  Months in sample: 3
  Classified as Crisis: 1
  Overlap: 33.3%
  Regime breakdown:
    {'Slowdown': 2, 'Crisis': 1}

Overall NBER overlap: 27.3% (gate: >= 80%)
  [FAIL] Crisis >= 80% of NBER recession months


### 4.6.3 Filtered vs Smoothed Probabilities

In [21]:
# Compare filtered (causal) vs smoothed (non-causal) probabilities
# This diagnostic shows how much information future data adds
diag_result, diag_summary = smoothed_vs_filtered_diagnostic(hmm_model, hmm_data)

print("Filtered vs Smoothed Diagnostic:")
print(f"  Mean KL divergence:         {diag_summary['mean_kl_divergence']:.6f}")
print(f"  Max KL divergence:          {diag_summary['max_kl_divergence']:.6f}")
print(f"  Classification agreement:   {diag_summary['classification_agreement']:.1%}")
print(f"\nInterpretation:")
print(f"  Low KL divergence => filtered probs are close to smoothed.")
print(f"  High agreement => state assignments rarely change with future info.")
print(f"  This is good: our filtered approach loses little information.")

logger.info(f"Filtered vs Smoothed: KL={diag_summary['mean_kl_divergence']:.6f}, "
            f"agreement={diag_summary['classification_agreement']:.1%}")

Filtered vs Smoothed Diagnostic:
  Mean KL divergence:         0.051977
  Max KL divergence:          1.236347
  Classification agreement:   96.2%

Interpretation:
  Low KL divergence => filtered probs are close to smoothed.
  High agreement => state assignments rarely change with future info.
  This is good: our filtered approach loses little information.


In [22]:
# Figure: KL divergence time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: filtered vs smoothed for crisis probability
# Map the crisis state: in reordered states, crisis = state 2
crisis_old_state = reorder[2]  # old state index corresponding to new state 2 (Crisis)
ax1.plot(composite_valid.index, diag_result[f'filtered_{crisis_old_state}'],
         label='Filtered (used)', color=COLORS['crisis'], linewidth=1)
ax1.plot(composite_valid.index, diag_result[f'smoothed_{crisis_old_state}'],
         label='Smoothed (forbidden)', color='grey', linewidth=1, linestyle='--', alpha=0.7)
ax1.set_ylabel('P(Crisis)')
ax1.set_title('Filtered vs Smoothed Crisis Probability', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: KL divergence
ax2.fill_between(composite_valid.index, diag_result['kl_divergence'],
                  alpha=0.5, color='steelblue')
ax2.plot(composite_valid.index, diag_result['kl_divergence'],
         color='steelblue', linewidth=0.5)
ax2.set_ylabel('KL Divergence')
ax2.set_xlabel('Date')
ax2.set_title('KL Divergence: Smoothed || Filtered', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'filtered_vs_smoothed_diagnostic')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/filtered_vs_smoothed_diagnostic.png


### 4.6.4 Regime Timeline

In [23]:
# Figure: Regime timeline with stacked probabilities
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [3, 1]},
                                sharex=True)

# Top panel: composite index with regime shading
ax1.plot(composite_valid.index, composite_valid.values, color='black', linewidth=0.8)

regime_color_map = {
    'Expansion': COLORS['expansion'],
    'Slowdown': COLORS['slowdown'],
    'Crisis': COLORS['crisis'],
}

for i in range(len(composite_valid) - 1):
    label = regime_probs['regime_label'].iloc[i]
    color = regime_color_map.get(label, '#cccccc')
    ax1.axvspan(composite_valid.index[i], composite_valid.index[i + 1],
               alpha=0.15, color=color, linewidth=0)

# Add NBER recession markers
for start, end in nber_recessions:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    if s >= composite_valid.index[0]:
        ax1.axvspan(s, e, alpha=0.08, color='black', linewidth=0, hatch='//')

ax1.axhline(y=0, color='grey', linestyle='--', alpha=0.5)
ax1.set_ylabel('Composite Z-Score')
ax1.set_title('Macro Regime Detection: Composite Index + HMM Regimes', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Create legend for regimes
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS['expansion'], alpha=0.3, label='Expansion'),
    Patch(facecolor=COLORS['slowdown'], alpha=0.3, label='Slowdown'),
    Patch(facecolor=COLORS['crisis'], alpha=0.3, label='Crisis'),
    Patch(facecolor='black', alpha=0.1, label='NBER Recession', hatch='//'),
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=9)

# Bottom panel: stacked regime probabilities
ax2.stackplot(
    regime_probs.index,
    regime_probs['p_expansion'],
    regime_probs['p_slowdown'],
    regime_probs['p_crisis'],
    colors=[COLORS['expansion'], COLORS['slowdown'], COLORS['crisis']],
    alpha=0.7,
    labels=['Expansion', 'Slowdown', 'Crisis'],
)
ax2.set_ylabel('Regime Probability')
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'regime_timeline_with_probabilities')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_timeline_with_probabilities.png


## 4.7 Validation Gates

In [24]:
# === VALIDATION GATES ===
print("=" * 60)
print("PHASE 4 VALIDATION GATES")
print("=" * 60)

gates = {}

# Gate 1: NBER overlap >= 80%
if total_recession_months > 0:
    gates['Crisis aligns with NBER (>=80%)'] = overall_overlap >= 0.80
else:
    gates['Crisis aligns with NBER (>=80%)'] = None  # Cannot evaluate

# Gate 2: Transition matrix diagonal > 0.7
gates['Transition diag > 0.7'] = all(d > 0.7 for d in diag)

# Gate 3: BIC 3-state < 2-state (or documented)
gates['BIC: 3-state < 2-state'] = bic_3 < bic_2

# Gate 4: Crisis < 20%
gates['Crisis < 20% of months'] = crisis_pct < 0.20

# Gate 5: Expansion > 40%
gates['Expansion > 40% of months'] = expansion_pct > 0.40

# Gate 6: Filtered probabilities used (by construction)
gates['Filtered probs used (not smoothed)'] = True  # Enforced by code

# Gate 7: HMM converged
gates['HMM converged'] = hmm_model.monitor_.converged

# Gate 8: Top-3 restarts consistent
if len(all_scores_sorted) >= 3:
    top3_spread = all_scores_sorted[0] - all_scores_sorted[2]
    gates['Top-3 LL consistent (spread<5)'] = top3_spread < 5
else:
    gates['Top-3 LL consistent (spread<5)'] = False

# Gate 9: PCA 1st component > 30% variance
gates['PCA PC1 > 30% variance'] = pc1_var > 0.30

for gate_name, passed in gates.items():
    if passed is None:
        status = 'N/A '
    elif passed:
        status = 'PASS'
    else:
        status = 'FAIL'
    print(f"  [{status}] {gate_name}")

n_evaluated = sum(1 for v in gates.values() if v is not None)
n_pass = sum(1 for v in gates.values() if v is True)
print(f"\nResult: {n_pass}/{n_evaluated} gates passed")

logger.info(f"Validation: {n_pass}/{n_evaluated} gates passed")

PHASE 4 VALIDATION GATES
  [FAIL] Crisis aligns with NBER (>=80%)
  [PASS] Transition diag > 0.7
  [PASS] BIC: 3-state < 2-state
  [FAIL] Crisis < 20% of months
  [FAIL] Expansion > 40% of months
  [PASS] Filtered probs used (not smoothed)
  [PASS] HMM converged
  [PASS] Top-3 LL consistent (spread<5)
  [PASS] PCA PC1 > 30% variance

Result: 4/9 gates passed


## 4.8 Export Outputs

In [25]:
# === Export all Phase 4 outputs ===

# 1. Regime probabilities (month-end)
regime_probs.to_parquet(REGIME_PROBS_FILE)
print(f"Exported regime_probabilities.parquet: {regime_probs.shape}")
print(f"  Date range: {regime_probs.index.min()} to {regime_probs.index.max()}")
print(f"  Columns: {list(regime_probs.columns)}")
print(f"  NaN count: {regime_probs.isna().sum().sum()}")

# 2. Macro composite index (month-end)
composite_df = composite_z.dropna().to_frame(name='composite_z')
composite_df.to_parquet(MACRO_COMPOSITE_FILE)
print(f"\nExported macro_composite_index.parquet: {composite_df.shape}")
print(f"  Date range: {composite_df.index.min()} to {composite_df.index.max()}")
print(f"  NaN count: {composite_df.isna().sum().sum()}")

# 3. Macro regimes for tech portfolio (daily)
macro_regimes_df = manual_regimes.to_frame(name='macro_regime')
macro_regimes_df.to_parquet(MACRO_REGIMES_FILE)
print(f"\nExported macro_regimes.parquet: {macro_regimes_df.shape}")
print(f"  Date range: {macro_regimes_df.index.min()} to {macro_regimes_df.index.max()}")
print(f"  Unique regimes: {macro_regimes_df['macro_regime'].unique().tolist()}")

# 4. Regime labels for tech portfolio (daily)
# Fill any remaining NaN
regime_labels_df = regime_labels_df.ffill().bfill()
regime_labels_df.to_parquet(REGIME_LABELS_FILE)
print(f"\nExported regime_labels.parquet: {regime_labels_df.shape}")
print(f"  Date range: {regime_labels_df.index.min()} to {regime_labels_df.index.max()}")
print(f"  Columns: {list(regime_labels_df.columns)}")
print(f"  NaN count: {regime_labels_df.isna().sum().sum()}")

# Save BIC results table
bic_results.to_csv(TABLES_DIR / 'hmm_bic_selection.csv', index=False)
print(f"\nExported hmm_bic_selection.csv")

# Save transition analysis table
analysis['transition_matrix'].to_csv(TABLES_DIR / 'hmm_transition_matrix.csv')
print(f"Exported hmm_transition_matrix.csv")

logger.info("Phase 4 complete -- all outputs exported")
print("\n=== Phase 4 Complete ===")

Exported regime_probabilities.parquet: (186, 4)
  Date range: 2008-11-30 00:00:00 to 2024-04-30 00:00:00
  Columns: ['p_expansion', 'p_slowdown', 'p_crisis', 'regime_label']
  NaN count: 0

Exported macro_composite_index.parquet: (186, 1)
  Date range: 2008-11-30 00:00:00 to 2024-04-30 00:00:00
  NaN count: 0

Exported macro_regimes.parquet: (2673, 1)
  Date range: 2016-01-01 00:00:00 to 2026-03-31 00:00:00
  Unique regimes: ['QE Era', 'Rate Hike', 'Late Cycle', 'COVID Crash', 'Zero-Rate Recovery', 'Inflation/Rate Shock', 'AI Boom', 'Tariff/Geopolitical']

Exported regime_labels.parquet: (2673, 4)
  Date range: 2016-01-01 00:00:00 to 2026-03-31 00:00:00
  Columns: ['regime_state', 'regime_prob_0', 'regime_prob_1', 'regime_prob_2']
  NaN count: 0

Exported hmm_bic_selection.csv
Exported hmm_transition_matrix.csv

=== Phase 4 Complete ===


In [26]:
# Final summary
print("=" * 60)
print("PHASE 4 SUMMARY")
print("=" * 60)
print(f"Composite index: {composite_df.shape[0]} monthly observations")
print(f"PCA PC1 variance: {pc1_var:.1%}")
print(f"HMM: {HMM_N_STATES}-state, {HMM_N_RESTARTS} restarts, converged={hmm_model.monitor_.converged}")
print(f"Best BIC model: K={int(best_k)}")
print(f"\nRegime proportions:")
for regime in ['Expansion', 'Slowdown', 'Crisis']:
    pct = regime_counts.get(regime, 0)
    print(f"  {regime}: {pct:.1%}")
print(f"\nTransition matrix diagonal: {diag.round(3)}")
print(f"Expected durations (months): {analysis['expected_duration_months'].round(1).to_dict()}")
print(f"\nFiltered vs Smoothed agreement: {diag_summary['classification_agreement']:.1%}")
print(f"\nOutputs saved:")
print(f"  - {REGIME_PROBS_FILE}")
print(f"  - {MACRO_COMPOSITE_FILE}")
print(f"  - {MACRO_REGIMES_FILE}")
print(f"  - {REGIME_LABELS_FILE}")
print(f"  - {FIGURES_DIR}/regime_timeline_with_probabilities.png")
print(f"  - {FIGURES_DIR}/transition_matrix_heatmap.png")
print(f"  - {FIGURES_DIR}/bic_model_selection.png")
print(f"  - {FIGURES_DIR}/pca_explained_variance.png")
print(f"  - {FIGURES_DIR}/composite_macro_index.png")
print(f"  - {FIGURES_DIR}/filtered_vs_smoothed_diagnostic.png")

PHASE 4 SUMMARY
Composite index: 186 monthly observations
PCA PC1 variance: 48.2%
HMM: 3-state, 25 restarts, converged=True
Best BIC model: K=3

Regime proportions:
  Expansion: 28.5%
  Slowdown: 12.9%
  Crisis: 58.6%

Transition matrix diagonal: [0.963 0.874 0.99 ]
Expected durations (months): {'Expansion': 26.7, 'Slowdown': 8.0, 'Crisis': 98.5}

Filtered vs Smoothed agreement: 96.2%

Outputs saved:
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/regime_probabilities.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/macro_composite_index.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/macro_regimes.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/regime_labels.parquet
  - /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_timel